# Sentiment Forecasting: FinBERT + Topological Data Analysis

A sophisticated approach to stock market sentiment analysis combining:

- **FinBERT**: State-of-the-art financial sentiment analysis
- **Topological Data Analysis (TDA)**: Persistent homology for market regime detection
- **ML Pipeline**: End-to-end backtesting and signal generation

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
from src.config import TICKERS, DEFAULT_LOOKBACK_DAYS, PREDICTION_THRESHOLD
from src.news import fetch_all_headlines
from src.sentiment import score_headlines, aggregate_daily_sentiment
from src.features import compute_price_features, join_sentiment_with_prices
from src.ml import add_quick_prob, train_model_from_frame, add_ml_prob, DEFAULT_FEATURES
from src.backtest import apply_signals, backtest_equal_weight, todays_signals
from src.visuals import plot_equity

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)

print(f"Analyzing tickers: {TICKERS[:5]}")

## 2. Live Sentiment Analysis with FinBERT

FinBERT is a BERT model fine-tuned on financial text. It classifies headlines as:
- **POSITIVE**: Bullish sentiment
- **NEGATIVE**: Bearish sentiment  
- **NEUTRAL**: No clear directional bias

In [ ]:
# Fetch recent headlines
tickers = ['SPY', 'QQQ', 'AAPL', 'NVDA', 'MSFT']
print(f"Fetching headlines for: {tickers}")

news = fetch_all_headlines(tickers, days=5)
print(f"\nFound {len(news)} headlines")
news.head(10)

In [ ]:
# Score headlines with FinBERT
print("Scoring with FinBERT (this may take a moment on first run)...")
scored = score_headlines(news)

# Show sample results
display_cols = ['ticker', 'headline', 'sent_label', 'sent_score']
scored[display_cols].head(10)

In [ ]:
# Visualize sentiment distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sentiment label distribution
label_counts = scored['sent_label'].value_counts()
colors = {'POSITIVE': '#2ecc71', 'NEGATIVE': '#e74c3c', 'NEUTRAL': '#95a5a6'}
axes[0].bar(label_counts.index, label_counts.values, 
            color=[colors.get(l, '#3498db') for l in label_counts.index])
axes[0].set_title('Sentiment Label Distribution')
axes[0].set_ylabel('Count')

# Sentiment score distribution
axes[1].hist(scored['sent_score'], bins=30, color='#3498db', edgecolor='white', alpha=0.7)
axes[1].axvline(x=0, color='red', linestyle='--', label='Neutral')
axes[1].set_title('Sentiment Score Distribution')
axes[1].set_xlabel('Sentiment Score (-1 to +1)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Topological Data Analysis (The "Wow" Factor)

### What is TDA?

**Topological Data Analysis** uses concepts from algebraic topology to analyze the "shape" of data. Key concepts:

- **Persistent Homology**: Tracks how topological features (connected components, loops, voids) appear and disappear as we vary a scale parameter
- **Delay Embedding**: Transforms a time series into a point cloud in higher dimensions
- **Persistence Diagrams**: Visual representation of topological features and their "lifetimes"

### Why TDA for Finance?

1. **Regime Detection**: Topological features can capture market regimes (trending, mean-reverting, volatile)
2. **Noise Robustness**: Persistent features are stable under small perturbations
3. **Non-linear Patterns**: Captures complex patterns that linear methods miss

In [ ]:
# Compute price features with topology
print("Computing price features with TDA (this may take a moment)...")
price_features = compute_price_features(tickers, lookback_days=180, add_topology=True)

# Show topology features
topo_cols = [c for c in price_features.columns if c.startswith('topo_')]
print(f"\nTopology features: {topo_cols}")
price_features[['ticker', 'date'] + topo_cols].tail(10)

In [ ]:
# Visualize topology features over time for one ticker
ticker = 'SPY'
spy_data = price_features[price_features['ticker'] == ticker].copy()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Price returns
axes[0].plot(spy_data['date'], spy_data['ret_5'], color='#3498db', alpha=0.7)
axes[0].set_ylabel('5-day Return')
axes[0].set_title(f'{ticker} - Returns and Topology Features')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)

# D0 persistence (connected components)
axes[1].plot(spy_data['date'], spy_data['topo_d0_total'], color='#e74c3c', alpha=0.7)
axes[1].set_ylabel('D0 Total Persistence')
axes[1].set_title('H0 (Connected Components) - Measures clustering in price space')

# Entropy
axes[2].plot(spy_data['date'], spy_data['topo_entropy'], color='#2ecc71', alpha=0.7)
axes[2].set_ylabel('Topo Entropy')
axes[2].set_title('Topological Entropy - Measures complexity of price dynamics')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()

## 4. Feature Engineering

We combine multiple feature types:
- **Price features**: Returns, volatility
- **Sentiment features**: FinBERT scores aggregated to daily level
- **Topology features**: Persistent homology summaries

In [ ]:
# Aggregate sentiment to daily level
daily_sentiment = aggregate_daily_sentiment(scored)
print(f"Daily sentiment records: {len(daily_sentiment)}")
daily_sentiment.head()

In [ ]:
# Join sentiment with price features
X = join_sentiment_with_prices(daily_sentiment, price_features)
print(f"\nJoined feature matrix: {X.shape}")
print(f"Columns: {list(X.columns)}")
X.tail(10)

In [ ]:
# Feature correlation heatmap
feature_cols = ['ret_1', 'ret_5', 'ret_10', 'vol_20', 'sent_mean', 'topo_d0_total', 'topo_entropy', 'label']
available_cols = [c for c in feature_cols if c in X.columns]

corr_matrix = X[available_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Model Training & Evaluation

We train a logistic regression model with:
- Time-based train/test split (no look-ahead bias)
- Balanced class weights
- Threshold optimization on training data

In [ ]:
# Train model
print("Training logistic regression model...")
model, report, threshold, features_used = train_model_from_frame(
    X, model_type='logistic', test_frac=0.2, calibrate=True
)

print(f"\nFeatures used: {features_used}")
print(f"Optimal threshold: {threshold:.3f}")
print(f"\nTest Set Performance:")
print(f"  Accuracy:  {report.accuracy:.3f}")
print(f"  Precision: {report.precision:.3f}")
print(f"  Recall:    {report.recall:.3f}")
print(f"  AUC:       {report.auc:.3f}")

In [ ]:
# Compare ML vs Rule-based approach
# Add ML probabilities
X_ml = add_ml_prob(X.copy(), model, features_used, out_col='p_up')

# Add rule-based probabilities
X_rule = add_quick_prob(X.copy())

print("Probability comparison:")
print(X_ml[['ticker', 'date', 'sent_mean', 'ret_5', 'p_up']].tail(10))

## 6. Backtesting Results

We run a simple equal-weight backtest:
- Go long on tickers where P(up) > threshold AND momentum confirms
- Equal weight across all longs
- Include trading costs

In [ ]:
# Run backtests
# ML-based
Xs_ml = apply_signals(X_ml, threshold=threshold, require_mom_agree=True)
perf_ml, metrics_ml = backtest_equal_weight(Xs_ml, cost_bps=3.0)

# Rule-based
Xs_rule = apply_signals(X_rule, threshold=0.55, require_mom_agree=True)
perf_rule, metrics_rule = backtest_equal_weight(Xs_rule, cost_bps=3.0)

print("Backtest Results:")
print(f"\nML-Based Strategy:")
print(f"  Sharpe Ratio: {metrics_ml['sharpe']:.2f}")
print(f"  Max Drawdown: {metrics_ml['max_dd']:.2%}")
print(f"  Trade Days:   {metrics_ml['trades']}")

print(f"\nRule-Based Strategy:")
print(f"  Sharpe Ratio: {metrics_rule['sharpe']:.2f}")
print(f"  Max Drawdown: {metrics_rule['max_dd']:.2%}")
print(f"  Trade Days:   {metrics_rule['trades']}")

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(12, 6))

if not perf_ml.empty:
    ax.plot(pd.to_datetime(perf_ml['date']), perf_ml['equity'], 
            label=f"ML-Based (Sharpe: {metrics_ml['sharpe']:.2f})", color='#3498db', linewidth=2)

if not perf_rule.empty:
    ax.plot(pd.to_datetime(perf_rule['date']), perf_rule['equity'], 
            label=f"Rule-Based (Sharpe: {metrics_rule['sharpe']:.2f})", color='#e74c3c', linewidth=2, alpha=0.7)

ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
ax.set_xlabel('Date')
ax.set_ylabel('Equity')
ax.set_title('Strategy Comparison: ML vs Rule-Based')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Today's signals
print("Today's Trading Signals:")
picks = todays_signals(X_ml, threshold=threshold, require_mom_agree=True)

if picks.empty:
    print("No signals today with current settings.")
else:
    display(picks)

## 7. Conclusions

### Key Findings

1. **FinBERT provides actionable sentiment**: Financial-specific NLP outperforms generic sentiment analysis

2. **TDA captures market regimes**: Topology features add information beyond price/momentum

3. **Combined approach**: Sentiment + Price + TDA features provide a rich feature set

### Future Improvements

- **More data sources**: Twitter, Reddit, earnings calls
- **Advanced models**: XGBoost ensemble, neural networks
- **Position sizing**: Volatility targeting, risk parity
- **Live trading**: API integration for paper trading

---

*This notebook is for educational purposes only. Not investment advice.*